
Fine Tuning roadmap:

- Prepare data: Convert your balanced DataFrame into a Hugging Face Dataset with text and numeric labels

- Tokenize: Run the tokenizer over all your texts so the model can read them
- Train/test split: Split the dataset (Hugging Face has a built-in method for this)
- Load model: Load roberta-base configured for 3-class classification
- Set training arguments: Things like learning rate, batch size, number of epochs
- Train: Use Hugging Face's Trainer to fine-tune the model on your data
- Evaluate: Run predictions on your test set, generate classification report and confusion matrix

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Tokenize

import pandas as pd

from transformers import AutoTokenizer

df_balanced = pd.read_csv("balanced_reviews.csv")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


In [ ]:
df_balanced = pd.read_csv("balanced_reviews.csv")
df_balanced.head()

In [ ]:
# convert your actual sentiment labels into numbers for the model

actual_map = {"negative": 0, "neutral": 1, "positive": 2}
df_balanced["actual_sentiment"] = df_balanced["rating_sentiment"].map(actual_map)
df_balanced.head()

In [ ]:
# convert the dataset from pandas to Dataset for HuggingFace model

from datasets import Dataset

dataset = Dataset.from_pandas(df_balanced[["reviews.text", "actual_sentiment"]])
dataset = dataset.rename_column("actual_sentiment", "labels")

In [ ]:
# tokenize the reviews for embedding

def tokenize_function(examples):
    return tokenizer(examples["reviews.text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
# train test split tokenized dataset with HuggingFace
split_tokenized_data = tokenized_dataset.train_test_split(test_size=0.3)

In [ ]:
# load the model
# The num_labels=3 tells it you have three classes (positive, negative, neutral)

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=3)

In [ ]:
# Set training arguments: Things like learning rate, batch size, number of epochs

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_strategy="epoch",
)

In [ ]:
from transformers import Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_tokenized_data["train"],
    eval_dataset=split_tokenized_data["test"],
    data_collator=data_collator
)

trainer.train()

In [ ]:
# run classificaiton report

from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(split_tokenized_data["test"])


In [ ]:
print(predictions)

In [ ]:
import numpy as np

y_pred = np.argmax(predictions.predictions, axis=1)
y_test = predictions.label_ids

print(classification_report(y_test, y_pred, target_names=["negative", "neutral", "positive"]))

In [ ]:
model.save_pretrained("./fine_tuned_roberta")
tokenizer.save_pretrained("./fine_tuned_roberta")